# 0330 · Base analítica v2: una fila por guía (propuesta P3)

**Qué hace.** Construye la base analítica con una fila por guía y un instante de predicción explícito: T0, la primera salida a zona urbana. Calcula las horas exactas (no `DATEDIFF`), la etiqueta de incumplimiento con estados de desenlace, el origen del SLA y el momento en que cada etiqueta queda conocida. Escribe un archivo **nuevo**, sin sobrescribir nada.

**Cuándo usarlo.** Solo **después** de que 0310 y 0320 estén entregados y se haya decidido la alternativa A (desenlace reconstruido con una fecha de entrega validada).

**Requisitos.** 0300 hasta la posición 10 y los Parquet de seguimiento, documentos y parametrías.

**Cómo usarlo.**
1. Completar la celda **«Parámetros por confirmar»** con lo acordado con el asesor y con Servienvios.
2. Ejecutar todo. La celda de controles **se detiene con error** si hay más filas que guías o etiquetas fuera de {0,1}; en ese caso no se sigue a 0410.
3. Entregar las tablas de control, el balance y el manifiesto.

In [ ]:
# Si el entorno no tiene DuckDB, quitar el comentario y ejecutar:
# %pip install duckdb

In [ ]:
# CONFIGURACIÓN · revisar las rutas antes de ejecutar (son las mismas de 0300)
import os
PATH_DATABRICKS = "/Volumes/proyectogrado/source/sourcedb/"
RUTA_GUIAS      = f"{PATH_DATABRICKS}LOG_GUIAS/RAW/LOG_GUIAS_BOGOTA.parquet"                 # la crea 0300, pos. 10
RUTA_MOVS_GLOB  = f"{PATH_DATABRICKS}LOG_SEGUIMIENTO_ENVIO/LOG_SEGUIMIENTO_ENVIO*.parquet"    # TODOS los movimientos
RUTA_DOCS_GLOB  = f"{PATH_DATABRICKS}LOG_DOCUMENTOS/LOG_DOCUMENTOS*.parquet"                 # TODOS los documentos
RUTA_TABLA_BASE = f"{PATH_DATABRICKS}LOG_CONSOLIDADO/LOG_BASE_BOGOTA.parquet"                # la crea 0300, pos. 30
PAR = f"{PATH_DATABRICKS}PARAMETRICAS/"
P = {"CIUDADES":   f"{PAR}PAR.CIUDADES.parquet",       "REGIONALES": f"{PAR}PAR.REGIONALES_PAIS.parquet",
     "EST_ENVIO":  f"{PAR}PAR.ESTADOSENVIO.parquet",   "EST_GUIA":   f"{PAR}PAR.ESTADOSGUIA.parquet",
     "RED":        f"{PAR}PAR.RED_OPERATIVA_OFERTA.parquet", "PROCESOS": f"{PAR}PAR.PROCESOS.parquet",
     "ZONAS":      f"{PAR}PAR.ZONASURBANAS.parquet"}
# Salidas nuevas de esta revisión: nada de lo existente se sobrescribe
CARPETA_SALIDAS = f"{PATH_DATABRICKS}LOG_CONSOLIDADO/REVISION_2026_10/"
os.makedirs(CARPETA_SALIDAS, exist_ok=True)
RUTA_GOLD_V2 = f"{CARPETA_SALIDAS}GOLD_V2_ENVIO.parquet"

## Parámetros por confirmar

Ninguno de estos valores está validado todavía. La celda siguiente propone un valor para el corte de observación a partir de la fecha de extracción, pero debe confirmarse.

In [ ]:
import duckdb
print(duckdb.sql(f"""SELECT MAX(FECHA_EXTRACCION_DW) AS ultima_extraccion_guias, MAX(FECHA_ENVIO) AS ultimo_envio
                      FROM read_parquet('{RUTA_GUIAS}')""").df())

In [ ]:
CAMPO_ENTREGA     = "FECHA_ENTREGA"        # columna de LOG_GUIAS validada en 0310 como entrega efectiva
INCUMPLE_SI_IGUAL = False                  # True si llegar exactamente al vencimiento cuenta como incumplimiento
CORTE_OBSERVACION = "2026-08-31 23:59:59"  # último instante observado en la extracción (confirmar)
ID_SALIDA_ZONA    = 9

## Funciones (no es necesario modificarlas)

In [ ]:
# =====================================================================================
# P3 · BASE ANALÍTICA V2: UNA FILA POR GUÍA, INSTANTE DE PREDICCIÓN EXPLÍCITO Y ETIQUETAS
#      CON REGLAS VERIFICABLES (alternativa A: desenlace reconstruido con una fecha de
#      entrega VALIDADA en la auditoría P1).
# Ubicación sugerida: reemplaza las posiciones 38–45 de 0300 (no las borra: se agrega como
# sección nueva y se escribe en archivos con nombre nuevo; nada se sobrescribe).
# Supuestos que NO están validados y se dejan como parámetros: CAMPO_ENTREGA,
# INCUMPLE_SI_IGUAL, CORTE_OBSERVACION, interpretación continua (no hábil) del SLA.
# =====================================================================================
import duckdb

def construir_gold_v2(R, P, RUTA_SALIDA, CAMPO_ENTREGA="FECHA_ENTREGA", ID_SALIDA_ZONA=9,
                      INCUMPLE_SI_IGUAL=False, CORTE_OBSERVACION="2026-08-31 23:59:59"):
    op = ">=" if INCUMPLE_SI_IGUAL else ">"
    con = duckdb.connect()
    con.execute(f"""
    COPY (
    WITH
    GUI AS (
        SELECT CAST(NUMERO_GUIA AS BIGINT) AS guia, ID_TIPOGUIA, ID_PAIS_ORIGEN, ID_CIUDAD_ORIGEN,
               ID_PAIS_DESTINO, ID_CIUDAD_DESTINO, ID_TIEMPO, ID_MEDTRANS, ID_PRODUCTO, ID_SUBPRODUCTO,
               ID_ESTADOGUIA, FECHA_ENVIO, {CAMPO_ENTREGA} AS fecha_entrega,
               NUMERO_PIEZAS AS piezas_declaradas, PESO_GUIA AS peso_declarado_kg,
               LARGO_ENVIO, ANCHO_ENVIO, ALTO_ENVIO
        FROM read_parquet('{R["GUIAS"]}')
    ),
    -- T0 = PRIMERA salida a zona urbana (desempate determinista); se conservan cuántas hubo
    SAL AS (
        SELECT CAST(NUMERO_GUIA AS BIGINT) AS guia, ID_PROCESO, NUMERO_MOVIMI, CONSECUT_MOVIMI,
               FECHA_HORA_MOVIMIENTO AS t0,
               COUNT(*) OVER (PARTITION BY CAST(NUMERO_GUIA AS BIGINT)) AS n_salidas_zona
        FROM read_parquet('{R["MOVS_GLOB"]}')
        WHERE ID_PROCESO = {ID_SALIDA_ZONA}
          AND CAST(NUMERO_GUIA AS BIGINT) IN (SELECT guia FROM GUI)   -- solo guías de la población
        QUALIFY ROW_NUMBER() OVER (PARTITION BY CAST(NUMERO_GUIA AS BIGINT)
                                   ORDER BY FECHA_HORA_MOVIMIENTO, NUMERO_MOVIMI, CONSECUT_MOVIMI) = 1
    ),
    -- Documento del movimiento T0, deduplicado por su clave (el conteo previo va en el control)
    DOC AS (
        SELECT ID_PROCESO, NUMERO_MOVIMI, CONSECUT_MOVIMI, ID_TIPO_ZONA_URBA, ID_CIUDAD_ZONA_URBA, ID_ZONA_URBA
        FROM read_parquet('{R["DOCS_GLOB"]}')
        QUALIFY ROW_NUMBER() OVER (PARTITION BY ID_PROCESO, NUMERO_MOVIMI, CONSECUT_MOVIMI
                                   ORDER BY FECHA_EXTRACCION_DW DESC, FECHA_MODIFICACION DESC) = 1
    ),
    -- Parametrías reducidas a UNA fila por clave; si la clave trae valores distintos, queda NULL y bandera
    RED AS (
        SELECT ID_CIUDAD_ORIGEN, ID_CIUDAD_DESTINO, ID_TIEMPO, ID_MEDTRANS,
               COUNT(*) AS n_red,
               CASE WHEN MIN(TIEMPO_ENTREGA_COMERCIAL_HOR) = MAX(TIEMPO_ENTREGA_COMERCIAL_HOR)
                    THEN MIN(TIEMPO_ENTREGA_COMERCIAL_HOR) END AS sla_h
        FROM read_parquet('{P["RED"]}') GROUP BY ALL
    ),
    ZON AS (
        SELECT ID_CIUDAD, ID_TIPO_ZONA_URBA, ID_ZONA_URBA, COUNT(*) AS n_zona,
               CASE WHEN COUNT(DISTINCT NOMBRE_ZONA_URBA) = 1 THEN MIN(NOMBRE_ZONA_URBA) END AS zona_urbana,
               CASE WHEN COUNT(DISTINCT CATEGORIA_ZONA_URBA) = 1 THEN MIN(CATEGORIA_ZONA_URBA) END AS categoria_zona
        FROM read_parquet('{P["ZONAS"]}') WHERE ID_PAIS = 1 GROUP BY ALL
    ),
    CIU AS (
        SELECT ID_PAIS, ID_CIUDAD, COUNT(*) AS n_ciu,
               CASE WHEN COUNT(DISTINCT NOMBRE_CIUDAD) = 1 THEN MIN(NOMBRE_CIUDAD) END AS ciudad,
               CASE WHEN COUNT(DISTINCT ID_REGIONAL)   = 1 THEN MIN(ID_REGIONAL)   END AS id_regional
        FROM read_parquet('{P["CIUDADES"]}') GROUP BY ALL
    ),
    REG AS (
        SELECT ID_REGIONAL, CASE WHEN COUNT(DISTINCT NOMBRE_REGIONAL) = 1 THEN MIN(NOMBRE_REGIONAL) END AS regional
        FROM read_parquet('{P["REGIONALES"]}') GROUP BY ALL
    ),
    BASE AS (
        SELECT G.*, S.t0, S.n_salidas_zona,
               D.ID_TIPO_ZONA_URBA, D.ID_ZONA_URBA,
               Z.zona_urbana, Z.categoria_zona, Z.n_zona,
               CO.ciudad AS ciudad_origen, RO.regional AS regional_origen,
               R2.sla_h, R2.n_red,
               (COALESCE(G.ID_TIEMPO,0) = 0 OR COALESCE(G.ID_MEDTRANS,0) = 0) AS sla_con_sustitucion_1
        FROM GUI G
        JOIN SAL S USING (guia)                                   -- población: guías con salida a zona
        LEFT JOIN DOC D ON D.ID_PROCESO = S.ID_PROCESO AND D.NUMERO_MOVIMI = S.NUMERO_MOVIMI
                       AND D.CONSECUT_MOVIMI = S.CONSECUT_MOVIMI
        LEFT JOIN ZON Z ON Z.ID_CIUDAD = D.ID_CIUDAD_ZONA_URBA AND Z.ID_TIPO_ZONA_URBA = D.ID_TIPO_ZONA_URBA
                       AND Z.ID_ZONA_URBA = D.ID_ZONA_URBA
        LEFT JOIN CIU CO ON CO.ID_PAIS = G.ID_PAIS_ORIGEN AND CO.ID_CIUDAD = G.ID_CIUDAD_ORIGEN
        LEFT JOIN REG RO ON RO.ID_REGIONAL = CO.id_regional
        LEFT JOIN RED R2 ON R2.ID_CIUDAD_ORIGEN = G.ID_CIUDAD_ORIGEN AND R2.ID_CIUDAD_DESTINO = G.ID_CIUDAD_DESTINO
                        AND R2.ID_TIEMPO   = CASE WHEN COALESCE(G.ID_TIEMPO,0)   <> 0 THEN G.ID_TIEMPO   ELSE 1 END
                        AND R2.ID_MEDTRANS = CASE WHEN COALESCE(G.ID_MEDTRANS,0) <> 0 THEN G.ID_MEDTRANS ELSE 1 END
    ),
    ETI AS (
        SELECT *,
            (epoch_ms(t0)            - epoch_ms(FECHA_ENVIO)) / 3600000.0 AS h_admision_a_t0,
            (epoch_ms(fecha_entrega) - epoch_ms(t0))          / 3600000.0 AS h_ultima_milla,
            (epoch_ms(fecha_entrega) - epoch_ms(FECHA_ENVIO)) / 3600000.0 AS h_total,
            epoch_ms(TIMESTAMP '{CORTE_OBSERVACION}')                       AS corte_ms,
            epoch_ms(FECHA_ENVIO) + CAST(sla_h * 3600000 AS BIGINT)         AS vencimiento_ms
        FROM BASE
    )
    SELECT * EXCLUDE (corte_ms, vencimiento_ms),
        make_timestamp(vencimiento_ms * 1000)                             AS vencimiento_sla,
        sla_h - h_admision_a_t0                                           AS margen_sla_en_t0_h,  -- predictor válido en T0
        CASE WHEN sla_h IS NULL THEN 'SIN_SLA'
             WHEN n_red > 1 THEN 'SLA_CLAVE_REPETIDA'
             WHEN sla_con_sustitucion_1 THEN 'SLA_CON_SUSTITUCION'
             ELSE 'SLA_PARAMETRIZADO' END                                 AS origen_sla,
        CASE WHEN fecha_entrega IS NOT NULL                    THEN 'ENTREGADO'
             WHEN vencimiento_ms IS NOT NULL AND vencimiento_ms < corte_ms THEN 'ABIERTO_VENCIDO'
             ELSE 'ABIERTO_NO_VENCIDO' END                                 AS estado_desenlace,
        -- Etiqueta: 1/0 cuando es identificable; NULL cuando no (abierto no vencido o sin SLA)
        CASE WHEN sla_h IS NULL                                THEN NULL
             WHEN fecha_entrega IS NOT NULL THEN (h_total {op} sla_h)::INT
             WHEN vencimiento_ms < corte_ms                     THEN 1
             ELSE NULL END                                                AS incumplimiento_sla,
        -- Momento en que cada etiqueta quedó conocida (para cortes y agregados sin futuro)
        make_timestamp(LEAST(epoch_ms(fecha_entrega), vencimiento_ms) * 1000) AS t_etiqueta_clf,
        fecha_entrega                                                     AS t_etiqueta_reg,
        (h_admision_a_t0 > sla_h)                                         AS vencido_antes_de_t0
    FROM ETI
    ) TO '{RUTA_SALIDA}' (FORMAT PARQUET, COMPRESSION ZSTD)
    """)
    con.close()

def controles_gold_v2(RUTA):
    """Pruebas de propiedades. Falla con mensaje claro; imprime solo agregados."""
    q = lambda s: duckdb.sql(s.replace("{T}", f"read_parquet('{RUTA}')")).df()
    c = q("""SELECT COUNT(*) filas, COUNT(DISTINCT guia) guias,
                    COUNT(*) FILTER (WHERE incumplimiento_sla NOT IN (0,1)) etiqueta_fuera_dominio,
                    COUNT(*) FILTER (WHERE incumplimiento_sla IS NULL) etiqueta_nula,
                    COUNT(*) FILTER (WHERE h_ultima_milla < 0) ultima_milla_negativa,
                    COUNT(*) FILTER (WHERE h_admision_a_t0 < 0) t0_antes_de_admision
             FROM {T}""")
    assert c.loc[0, "filas"] == c.loc[0, "guias"], "Hay más filas que guías: fan-out sin resolver"
    assert c.loc[0, "etiqueta_fuera_dominio"] == 0, "incumplimiento_sla fuera de {0,1}"
    tablas = {
        "control": c,
        "etiqueta_por_estado": q("""SELECT estado_desenlace, origen_sla, COUNT(*) guias,
                                      AVG(incumplimiento_sla) prevalencia, COUNT(incumplimiento_sla) n_etiquetadas
                               FROM {T} GROUP BY ALL ORDER BY guias DESC"""),
        "por_mes": q("""SELECT date_trunc('month', FECHA_ENVIO) mes, COUNT(*) guias,
                          AVG((estado_desenlace='ENTREGADO')::INT) pct_entregado,
                          AVG(incumplimiento_sla) prevalencia
                        FROM {T} GROUP BY 1 ORDER BY 1"""),
    }
    return tablas

## Construcción y controles

In [ ]:
R = {"GUIAS": RUTA_GUIAS, "MOVS_GLOB": RUTA_MOVS_GLOB, "DOCS_GLOB": RUTA_DOCS_GLOB}
construir_gold_v2(R, P, RUTA_GOLD_V2, CAMPO_ENTREGA=CAMPO_ENTREGA, ID_SALIDA_ZONA=ID_SALIDA_ZONA,
                  INCUMPLE_SI_IGUAL=INCUMPLE_SI_IGUAL, CORTE_OBSERVACION=CORTE_OBSERVACION)
tablas = controles_gold_v2(RUTA_GOLD_V2)     # se detiene con error si falla la unidad o el dominio
for nombre, t in tablas.items():
    print(nombre); display(t); t.to_csv(f"{CARPETA_SALIDAS}P3_{nombre}.csv", index=False)

In [ ]:
# Balance de registros: de LOG_GUIAS_BOGOTA a la base v2 (solo conteos)
balance = duckdb.sql(f"""
    SELECT 'Guías en LOG_GUIAS_BOGOTA' AS etapa, COUNT(*) AS guias FROM read_parquet('{RUTA_GUIAS}')
    UNION ALL SELECT 'Base v2: con salida a zona urbana', COUNT(*) FROM read_parquet('{RUTA_GOLD_V2}')
    UNION ALL SELECT '  con SLA parametrizado', COUNT(*) FROM read_parquet('{RUTA_GOLD_V2}') WHERE origen_sla = 'SLA_PARAMETRIZADO'
    UNION ALL SELECT '  con etiqueta identificable', COUNT(incumplimiento_sla) FROM read_parquet('{RUTA_GOLD_V2}')
    UNION ALL SELECT '  ya vencidas antes de T0', COUNT(*) FILTER (WHERE vencido_antes_de_t0) FROM read_parquet('{RUTA_GOLD_V2}')
    UNION ALL SELECT '  última milla negativa (auditar)', COUNT(*) FILTER (WHERE h_ultima_milla < 0) FROM read_parquet('{RUTA_GOLD_V2}')
""").df()
display(balance); balance.to_csv(f"{CARPETA_SALIDAS}P3_balance.csv", index=False)

# Documentos repetidos por clave en la fuente (la base v2 se queda con uno por clave)
display(duckdb.sql(f"""SELECT COUNT(*) AS claves_documento_repetidas FROM (
    SELECT ID_PROCESO, NUMERO_MOVIMI, CONSECUT_MOVIMI FROM read_parquet('{RUTA_DOCS_GLOB}')
    GROUP BY ALL HAVING COUNT(*) > 1)""").df())

In [ ]:
# Manifiesto de la corrida: entradas, parámetros, versiones y conteos
import json, glob, sys, datetime
def archivos(patron):
    return [{"archivo": f, "bytes": os.path.getsize(f), "modificado": datetime.datetime.fromtimestamp(os.path.getmtime(f)).isoformat()}
            for f in sorted(glob.glob(patron))]
manifiesto = {
    "fecha_corrida": datetime.datetime.now().isoformat(), "python": sys.version.split()[0], "duckdb": duckdb.__version__,
    "entradas": {"guias": archivos(RUTA_GUIAS), "movimientos": archivos(RUTA_MOVS_GLOB), "documentos": archivos(RUTA_DOCS_GLOB),
                 "parametricas": {k: archivos(v) for k, v in P.items()}},
    "parametros": {"CAMPO_ENTREGA": CAMPO_ENTREGA, "INCUMPLE_SI_IGUAL": INCUMPLE_SI_IGUAL,
                   "CORTE_OBSERVACION": CORTE_OBSERVACION, "ID_SALIDA_ZONA": ID_SALIDA_ZONA},
    "salida": RUTA_GOLD_V2, "control": tablas["control"].to_dict("records"), "balance": balance.to_dict("records")}
with open(f"{CARPETA_SALIDAS}P3_manifiesto.json", "w") as f:
    json.dump(manifiesto, f, indent=2, default=str)
print("Manifiesto guardado")

## Cómo leer los resultados

- `control`: `filas` debe ser igual a `guias` y `etiqueta_fuera_dominio` debe ser 0 (si no, la celda ya se detuvo). `ultima_milla_negativa` y `t0_antes_de_admision` se auditan, no se borran sin causa.
- `etiqueta_por_estado`: prevalencia y número de guías etiquetadas por estado del desenlace y origen del SLA. El análisis principal usa solo `SLA_PARAMETRIZADO`; el resto va como sensibilidad.
- `por_mes`: si los últimos meses tienen menos entregadas, hay envíos sin madurar; eso condiciona los cortes de 0410.
- `balance`: cuántas guías se quedan en cada paso.

## Qué entregar

Los CSV `P3_*` y `P3_manifiesto.json`. Con los controles aprobados se pasa a 0410.